#Importação dos Dados

In [ ]:
# Tratamento dos dados
import numpy as np
import pandas as pd

# Modelos de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import plot_tree

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Métricas de avaliação
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix, classification_report

# Plot dos gráficos
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns


# Recebendo os dados:
from googledrivedownloader import download_file_from_google_drive as gdd

In [ ]:

# Recebendo os dados:
data_google_id = '1ND6YYlWajD7UOEErmkb8NzZBkTPSU2lN'
gdd(file_id=data_google_id,
    dest_path = './dados.csv', # Faz o download dos dados e salva o mesmo num arquivo nomeado data.csv
    showsize = True)

# Armazenandos os dados em um DataFrame
# para receber os dados o sep teve como argumento o valor ','. Isso ocorreu devido a
dados = pd.read_csv("dados.csv", sep = ',')

# Exploração dos Dados

Análise exploratória dos dados


In [ ]:
dados.head()

,nivel_ruido,gases_toxicos,horas_turno,iluminacao,epis_usado,setor,clima,risco
0,97.0,8.831141,8.0,513.0,não,pintura,moderado,alto
1,76.0,30.308022,10.0,174.0,não,montagem,moderado,alto
2,95.0,39.436406,9.0,761.0,sim,pintura,frio,alto
3,97.0,37.261701,11.0,207.0,sim,montagem,moderado,alto
4,82.0,45.526084,6.0,323.0,não,soldagem,moderado,alto


In [ ]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   nivel_ruido    4725 non-null   float64
 1   gases_toxicos  4769 non-null   float64
 2   horas_turno    4756 non-null   float64
 3   iluminacao     4761 non-null   float64
 4   epis_usado     4768 non-null   object 
 5   setor          4727 non-null   object 
 6   clima          4769 non-null   object 
 7   risco          4769 non-null   object 
dtypes: float64(4), object(4)
memory usage: 312.6+ KB


Verificação de valores nulos

In [ ]:
dados.isnull().sum()

,0
nivel_ruido,275
gases_toxicos,231
horas_turno,244
iluminacao,239
epis_usado,232
setor,273
clima,231
risco,231


Geração de estatísticas descritivas (média, desvio, mínimo/máximo).

In [ ]:
dados.describe()

,nivel_ruido,gases_toxicos,horas_turno,iluminacao
count,4725.000000,4769.000000,4756.00000,4761.000000
mean,94.601693,25.085388,8.45164,553.387734
std,14.486953,14.555502,1.70049,261.088620
min,70.000000,0.051556,6.00000,100.000000
25%,82.000000,12.332626,7.00000,327.000000
50%,95.000000,25.255774,8.00000,558.000000
75%,107.000000,37.924321,10.00000,782.000000
max,119.000000,49.990289,11.00000,999.000000


#Tratamento dos dados

In [ ]:
for column in dados.columns:
    if dados[column].dtype == 'object':
        dados[column].fillna(dados[column].mode()[0], inplace=True)
    else:
        dados[column].fillna(dados[column].median(), inplace=True)

dados.isnull().sum()

/tmp/ipython-input-2070342502.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dados[column].fillna(dados[column].median(), inplace=True)
/tmp/ipython-input-2070342502.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True

,0
nivel_ruido,0
gases_toxicos,0
horas_turno,0
iluminacao,0
epis_usado,0
setor,0
clima,0
risco,0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dados['epis_usado_enc'] = dados['epis_usado'].map({'sim':1,'não':0})

In [ ]:
dados['risco_enc'] = dados['risco'].map({'alto':1,'baixo':0})

In [ ]:
dados = pd.get_dummies(dados, columns=['setor','clima'], drop_first=False)

In [ ]:
y = dados['risco_enc']
X = dados.drop(columns=['risco','epis_usado'], inplace=True)
dados.head()

,nivel_ruido,gases_toxicos,horas_turno,iluminacao,epis_usado_enc,risco_enc,setor_montagem,setor_pintura,setor_soldagem,clima_frio,clima_moderado,clima_quente
0,97.0,8.831141,8.0,513.0,0,1,False,True,False,False,True,False
1,76.0,30.308022,10.0,174.0,0,1,True,False,False,False,True,False
2,95.0,39.436406,9.0,761.0,1,1,False,True,False,True,False,False
3,97.0,37.261701,11.0,207.0,1,1,True,False,False,False,True,False
4,82.0,45.526084,6.0,323.0,0,1,False,False,True,False,True,False


In [ ]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   nivel_ruido     5000 non-null   float64
 1   gases_toxicos   5000 non-null   float64
 2   horas_turno     5000 non-null   float64
 3   iluminacao      5000 non-null   float64
 4   epis_usado_enc  5000 non-null   int64  
 5   risco_enc       5000 non-null   int64  
 6   setor_montagem  5000 non-null   bool   
 7   setor_pintura   5000 non-null   bool   
 8   setor_soldagem  5000 non-null   bool   
 9   clima_frio      5000 non-null   bool   
 10  clima_moderado  5000 non-null   bool   
 11  clima_quente    5000 non-null   bool   
dtypes: bool(6), float64(4), int64(2)
memory usage: 263.8 KB


#Treinamento dos modelos e métricas de desempenho

#Conclusões da atividade e Qualidade do trabalho

#Utilização do modelo salvo